In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

import os

In [2]:
def merge_csv_by_param_3(input_dir='dataset_Csv', output_dir='merged_data'):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    output_1500 = output_dir / 'data_1500.csv'
    output_2900 = output_dir / 'data_2900.csv'

    # eсли файлы уже существуют от прошлого запуска удаляем
    if output_1500.exists():
        output_1500.unlink()
    if output_2900.exists():
        output_2900.unlink()

    rows_1500 = 0
    rows_2900 = 0
    wrote_header_1500 = False
    wrote_header_2900 = False

    csv_files = list(input_dir.rglob('*.csv'))
    print(f"Найдено CSV файлов: {len(csv_files)}")

    rename_1500_dict = None
    rename_2900_dict = None

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)

            if rename_1500_dict is None:
                rename_1500_dict = {col: col for col in df.columns}
            if rename_2900_dict is None:
                rename_2900_dict = {col: col for col in df.columns}

            if 'center' in df.columns:
                param_center = df['center'].iloc[0]

                if param_center == 1500:
                    df = df.rename(columns=rename_1500_dict)
                    
                    df.to_csv(output_1500, mode='a', index=False, header=not wrote_header_1500)
                    wrote_header_1500 = True
                    rows_1500 += len(df)
                    print(f"  + {csv_file.name} -> 1500")

                elif param_center == 2900:
                    df = df.rename(columns=rename_2900_dict)
                    
                    df.to_csv(output_2900, mode='a', index=False, header=not wrote_header_2900)
                    wrote_header_2900 = True
                    rows_2900 += len(df)
                    print(f"  + {csv_file.name} -> 2900")

                else:
                    print(f"  ? {csv_file.name} -> {param_center} (пропущен)")
            else:
                print(f"  ! {csv_file.name} (нет колонки center)")

        except Exception as e:
            print(f"  ✗ Ошибка при чтении {csv_file.name}: {e}")

    if rows_1500 > 0:
        print(f"\nСохранено {rows_1500} строк в {output_1500}")
        df_1500 = pd.read_csv(output_1500, low_memory=False)
    else:
        print("\nНет данных для center = 1500")
        df_1500 = pd.DataFrame()

    if rows_2900 > 0:
        print(f"Сохранено {rows_2900} строк в {output_2900}")
        df_2900 = pd.read_csv(output_2900, low_memory=False)
    else:
        print("Нет данных для center = 2900")
        df_2900 = pd.DataFrame()

    return df_1500, df_2900

In [3]:
def analyze_14th_column_stats(input_dir='dataset_Csv'):
    input_dir = Path(input_dir)

    # Счетчик для названий 14-го столбца
    column_counter = Counter()
    total_files = 0

    csv_files = list(input_dir.rglob('*.csv'))
    print(f"Анализ {len(csv_files)} файлов...")

    for csv_file in csv_files:
        try:
            #читаем только заголовок для экономии памяти
            df_header = pd.read_csv(csv_file, nrows=0, low_memory=False)

            if len(df_header.columns) >= 16:
                # Получаем название 14-й колонки (индекс 13)
                col_14_name = df_header.columns[15]
                column_counter[col_14_name] += 1
            else:
                print(f"  ! {csv_file.name}: только {len(df_header.columns)} колонок")

            total_files += 1

        except Exception as e:
            print(f"  ✗ Ошибка при чтении {csv_file.name}: {e}")

    stats_data = []
    for col_name, count in column_counter.most_common():
        percentage = (count / total_files) * 100
        stats_data.append({
            'column_name': col_name,
            'count': count,
            'percentage': f'{percentage:.2f}%'
        })

    df_stats = pd.DataFrame(stats_data)

    print(f"\nСтатистика по 14-му столбцу:")
    print(f"Всего проанализировано файлов: {total_files}")
    print(f"Уникальных названий 14-го столбца: {len(column_counter)}")

    return df_stats

In [4]:
os.chdir("D:/hakaton")
analyze_14th_column_stats('datasetCsv')

Анализ 237 файлов...

Статистика по 14-му столбцу:
Всего проанализировано файлов: 237
Уникальных названий 14-го столбца: 6


,column_name,count,percentage
0,Wave_2459.8,59,24.89%
1,Wave_928.0,53,22.36%
2,Wave_2459.9,52,21.94%
3,Wave_927.9,49,20.68%
4,Wave_927.8,17,7.17%
5,Wave_2459.7,7,2.95%


In [5]:
os.chdir("D:/hakaton")

if __name__ == "__main__":

    df_1500, df_2900 = merge_csv_by_param_3('datasetCsv', 'mergedData')

    if not df_1500.empty:
        print(f"\nРазмерность данных для 1500: {df_1500.shape}")
    if not df_2900.empty:
        print(f"Размерность данных для 2900: {df_2900.shape}")

Найдено CSV файлов: 237
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_1.csv -> 1500
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place4_2.csv -> 1500
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_1.csv -> 1500
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place5_2.csv -> 1500
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_1.csv -> 1500
  + cortex_control_1group_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place6_2.csv -> 1500
  + cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place4_1.csv -> 2900
  + cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place4_2.csv -> 2900
  + cortex_control_1group_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place5_1.csv -> 2900
  + cortex_control_1group_633nm_center2900_obj100

In [6]:
os.chdir("D:/hakaton")
df = pd.read_csv("mergedData/data_1500.csv", low_memory=False)

df.head()

,position,class,group,nm,center,obj,power,during,acc,map,...,Wave_1993.8,Wave_1994.7,Wave_1995.7,Wave_1996.7,Wave_1997.6,Wave_1998.6,Wave_1999.5,Wave_2000.5,Wave_2001.5,Wave_2002.4
0,cortex,control,1,633,1500,100,100,1s,5,35x15,...,12658.952148,12875.486328,13037.258789,12694.952148,12942.969727,13076.103516,12984.517578,12979.162109,13013.024414,12803.853516
1,cortex,control,1,633,1500,100,100,1s,5,35x15,...,10330.268555,10418.412109,10545.763672,10594.799805,10573.313477,10632.827148,10415.318359,10281.377930,10419.309570,10437.611328
2,cortex,control,1,633,1500,100,100,1s,5,35x15,...,9753.319336,9773.462891,10046.941406,9724.959961,9988.084961,10023.967773,9903.046875,10035.650391,10021.884766,9919.914063
3,cortex,control,1,633,1500,100,100,1s,5,35x15,...,9165.926758,9102.402344,9370.529297,9348.813477,9175.556641,9305.356445,9176.457031,9251.411133,9098.919922,9245.338867
4,cortex,control,1,633,1500,100,100,1s,5,35x15,...,8899.642578,8846.510742,8950.057617,8938.708984,8885.554688,9030.978516,8860.208008,8825.307617,8918.510742,8936.812500


In [7]:
df = pd.read_csv("mergedData/data_2900.csv", low_memory=False)

df.head()

,position,class,group,nm,center,obj,power,during,acc,map,...,Wave_3281.5,Wave_3282.2,Wave_3283.0,Wave_3283.7,Wave_3284.5,Wave_3285.2,Wave_3286.0,Wave_3286.7,Wave_3287.4,Wave_3288.2
0,cortex,control,1,633,2900,100,100,1s,5,35x15,...,6350.583008,6324.770020,6325.937988,6343.979004,6240.523926,6133.655273,6330.616699,6426.342285,6251.896973,6238.386719
1,cortex,control,1,633,2900,100,100,1s,5,35x15,...,5898.656250,5457.854492,5705.152832,5591.475098,5548.632813,5600.293945,5581.071289,5484.172363,5481.809082,5549.360840
2,cortex,control,1,633,2900,100,100,1s,5,35x15,...,5699.673340,5741.204590,5802.993652,5820.938477,5646.509766,5694.813477,5891.693848,5781.344238,5951.292480,5718.239746
3,cortex,control,1,633,2900,100,100,1s,5,35x15,...,6289.875977,6321.396484,6406.910156,6350.728027,6291.150391,6349.700195,6337.369141,6365.557129,6292.428223,6451.174316
4,cortex,control,1,633,2900,100,100,1s,5,35x15,...,6974.511719,6844.244141,7041.190918,6806.279785,6810.912598,6832.426270,6951.860840,7030.816895,7180.730957,7133.444824
